# Adapter experiment: α sweeps, statement types, baselines

Re-runs the trained-adapter α sweeps through `shared_truth` and checks them against the published
results. **Nothing is uploaded:** every re-run writes to `OUT_DIR`, and the saved results on the Hub
are only read.

| part | needs | does |
|---|---|---|
| **1. Sweeps** | GPU | re-sweeps `RERUN_IDS` from a run set and compares them item by item with the saved `sweep_results.json` |
| **2. Tables A–D** | CPU | statement-type tables from the 12 full-test-set runs, checked against the published CSVs |
| **3. Baselines** *(optional)* | GPU | the Mse8 random-weight and shuffled baselines plus the activation dumps (old Cell 11) |
| **4. Tables E/F** *(with 3)* | CPU | baseline and residual-geometry tables from the Part 3 outputs |

**Run sets.** Each set is pinned by `pair_id`, and each run's adapter repo and subfolder are read
from its own saved payload, so any published run can be re-swept without the old config cells.

| `RUN_SET` | runs | eval |
|---|---|---|
| `published` | the 22 trajectories behind Table 7 and the bootstrap (one is a known duplicate) | simple statements, n=1599 |
| `multiseed_simple` | orig + seeds 42–44 on 3 trajectories | simple statements, n=1599 |
| `multiseed_full` | the same 12 adapters | full test set, n=2684 |

The default re-sweeps one run, the floor pair (Gemma-2-2B → Qwen2.5-1.5B, `stmt_origadapter`), as the
regression check.

In [1]:
# --- setup ------------------------------------------------------------------
import os, sys, subprocess
from pathlib import Path

BRANCH   = "rebuttal"
REPO_DIR = Path("/content/shared-truth")
FORK_DIR = Path("/content/Closing-Backdoors-Via-Representation-Transfer")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/KingTechnician/shared-truth.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
if not FORK_DIR.exists():   # ModelWrapper / AutoEncoder; needed by the GPU parts
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/withmartian/Closing-Backdoors-Via-Representation-Transfer.git",
                    str(FORK_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.append(str(FORK_DIR))

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))   # gated models (Llama, Gemma)
except Exception:
    pass

# Pin scikit-learn to the version the probes were saved with
from shared_truth import env
env.pin_sklearn()

import json, numpy as np, pandas as pd
from shared_truth import storage, metrics, naming, residuals
print("shared_truth @", storage.provenance()["shared_truth_sha"])

model.skops:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

installing scikit-learn==1.8.0 skops (probes were saved with scikit-learn 1.8.0; runtime had 1.6.1)
scikit-learn 1.8.0 (matches probes)
shared_truth @ db7fa277d8371a6fb6abf0c35191b6d6398e18f0


In [2]:
# --- config -----------------------------------------------------------------
RUN_SET   = "multiseed_full"      # "published" | "multiseed_simple" | "multiseed_full"
RERUN_IDS = [                     # re-sweep these (GPU); [] to skip Part 1, "all" for the whole set
    "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_origadapter",
]
ITEM_TOL  = 0                     # predictions allowed to differ at any α before Part 1 fails

RUN_BASELINES = True             # Part 3 (GPU, ~10 min per trajectory)
BASELINE_TRAJ = None              # None = all 3, or a subset of residuals.TRAJECTORY_ADAPTERS

OUT_DIR = Path("/content/adapter_rerun")   # scratch; never uploaded
PLOT    = True

In [3]:
# --- data root, run set, specs ----------------------------------------------
ROOT = Path(storage.fetch_results(include_analysis=True, include_dumps=RUN_BASELINES))
SWEEPS = ROOT / storage.RESULTS_PREFIX

def _published_ids():
    ref = json.load(open(ROOT / "strict_overshoot_bootstrap.json"))
    return sorted(r["pair_id"] for r in ref["per_trajectory"])

RUN_SETS = {   # name -> (pair_ids, statement_type filter, expected n)
    "published":        (_published_ids(), "simple", 1599),
    "multiseed_simple": (residuals.run_ids(residuals.SIMPLE_TRAINED_VARIANTS), "simple", 1599),
    "multiseed_full":   (residuals.run_ids(residuals.TRAINED_VARIANTS), None, 2684),
}
IDS, STATEMENT_TYPE, EXPECTED_N = RUN_SETS[RUN_SET]

missing = [p for p in IDS if not (SWEEPS / p / "sweep_results.json").exists()]
if missing:
    raise SystemExit("missing sweep_results.json:\n  " + "\n  ".join(missing))
saved = {p: json.load(open(SWEEPS / p / "sweep_results.json")) for p in IDS}
wrong_n = {p: s["n_test"] for p, s in saved.items() if s["n_test"] != EXPECTED_N}
if wrong_n:
    raise SystemExit(f"{RUN_SET} should be n={EXPECTED_N}; these are not: {wrong_n}")

SPEC = {p: naming.adapter_from_payload(saved[p]) for p in IDS}   # (repo, subfolder, variant, revision)
RERUN = list(IDS) if RERUN_IDS == "all" else list(RERUN_IDS)
outside = [p for p in RERUN if p not in SPEC]
if outside:
    raise SystemExit(f"RERUN_IDS not in {RUN_SET}: {outside}")

print(f"{RUN_SET}: {len(IDS)} runs, eval n={EXPECTED_N}, re-sweeping {len(RERUN)}")
for p in IDS:
    repo, sub, v, rev = SPEC[p]
    print(f"  {'→ ' if p in RERUN else '  '}{saved[p]['display_label']:<44} {v:<18} {repo.split('/')[-1]}"
          f"  @{rev[:10] if rev else 'unpinned'}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 88 files:   0%|          | 0/88 [00:00<?, ?it/s]

multiseed_full: 12 runs, eval n=2684, re-sweeping 1
    Gemma-2-2B (L13) → Llama-3.2-3B (L12)        stmt_origadapter   gemma_2_2b_instruct_l13_to_llama_3.2_3b_instruct_l12
    Gemma-2-2B (L13) → Llama-3.2-3B (L12)        stmt_seed42        gemma_2_2b_instruct_l13_to_llama_3.2_3b_instruct_l12_seed_42
    Gemma-2-2B (L13) → Llama-3.2-3B (L12)        stmt_seed43        gemma_2_2b_instruct_l13_to_llama_3.2_3b_instruct_l12_seed_43
    Gemma-2-2B (L13) → Llama-3.2-3B (L12)        stmt_seed44        gemma_2_2b_instruct_l13_to_llama_3.2_3b_instruct_l12_seed_44
    Llama-3.2-3B (L12) → Gemma-2-2B (L13)        stmt_origadapter   llama_3.2_3b_instruct_l12_to_gemma_2_2b_instruct_l13
    Llama-3.2-3B (L12) → Gemma-2-2B (L13)        stmt_seed42        llama_3.2_3b_instruct_l12_to_gemma_2_2b_instruct_l13_seed_42
    Llama-3.2-3B (L12) → Gemma-2-2B (L13)        stmt_seed43        llama_3.2_3b_instruct_l12_to_gemma_2_2b_instruct_l13_seed_43
    Llama-3.2-3B (L12) → Gemma-2-2B (L13)        stmt_seed44 

## Part 1 — re-sweep and compare (GPU)

Each run in `RERUN_IDS` is swept with `sweep.run_all` into `OUT_DIR`, then compared with its saved
`sweep_results.json` at every α:

- **items differing**: predictions that changed. This is exact where both sides store per-item
  `preds` (the full-test-set and multi-seed runs). For the older published runs, it is the accuracy
  change in items, which is a lower bound.
- **max |Δscore|** over all per-item probe scores, and **|ΔAUROC|**.
- the native ceilings, and strict overshoot old vs new.

fp16 matrix multiplies can differ in the last bit across GPU types, which could flip an item sitting on
the probe's threshold. `ITEM_TOL` sets how many flips are tolerated. The Procrustes recompute on the
same Colab GPU type matched with 0.

**Adapter revisions.** Before re-sweeping, each run's adapter is checked with `storage.verify_adapter`:
the repo must still serve the weights the run used (its recorded `config.json`, training metrics
included). The re-sweep is then pinned to that commit, and the new payload records it. Published runs
recorded no revision, so they are checked against HEAD; a run whose repo has since been retrained
(the floor pair's simple-statement seeds) stops here instead of silently sweeping other weights.

In [4]:
checks = {}
if not RERUN:
    print("RERUN_IDS is empty — skipping Part 1")
else:
    from shared_truth import sweep
    for p in RERUN:   # the repo must still serve this run's weights; pin the re-sweep to that commit
        try:
            sha = storage.verify_adapter(saved[p])
        except storage.AdapterMismatchError as err:
            raise SystemExit(f"cannot re-sweep {p}: {err}")
        SPEC[p] = SPEC[p][:3] + (sha,)
        print(f"  adapter verified: {p} @ {sha[:10]}")
    specs = [sweep.PairSpec(*SPEC[p]) for p in RERUN]
    new = sweep.run_all(specs, OUT_DIR, statement_type=STATEMENT_TYPE, plot=PLOT)

    for p in RERUN:
        a, b = new[p], saved[p]
        if a["n_test"] != b["n_test"]:
            raise SystemExit(f"{p}: re-run evaluated {a['n_test']} items, saved {b['n_test']}")
        d = metrics.sweep_diff(a["alpha_sweep"], b["alpha_sweep"])
        ceil = {side: (a["ceiling"][side]["accuracy"] - b["ceiling"][side]["accuracy"]) * a["n_test"]
                for side in ("source", "target")}
        so_new, so_old = (metrics.strict_overshoot(x["alpha_sweep"])[0] for x in (a, b))
        worst = max(r["items_differing"] for r in d)
        checks[p] = {"max_items_differing": worst,
                     "max_abs_dscore": max(r["max_abs_dscore"] for r in d),
                     "max_abs_dauroc": max(r["abs_dauroc"] for r in d),
                     "ceiling_items": ceil, "strict_new": so_new, "strict_saved": so_old,
                     "items_exact": all(r["items_exact"] for r in d)}
        print(f"\n=== {a['display_label']}  [{a['variant']}] ===")
        print(f"  items differing at worst α: {worst}"
              f"{'' if checks[p]['items_exact'] else ' (lower bound: saved run has no per-item preds)'}")
        print(f"  max |Δscore| {checks[p]['max_abs_dscore']:.2e}   max |ΔAUROC| {checks[p]['max_abs_dauroc']:.2e}")
        print(f"  ceiling change (items): source {ceil['source']:+.0f}, target {ceil['target']:+.0f}")
        print(f"  strict overshoot: re-run {so_new:+.2f} pp, saved {so_old:+.2f} pp")

    bad = {p: c["max_items_differing"] for p, c in checks.items()
           if c["max_items_differing"] > ITEM_TOL or any(abs(v) > ITEM_TOL for v in c["ceiling_items"].values())}
    if bad:
        raise SystemExit(f"REGRESSION — more than {ITEM_TOL} item(s) differ: {bad}")
    print(f"\n✓ {len(checks)} re-swept run(s) match the saved sweeps (≤ {ITEM_TOL} items at every α)")

run  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_origadapter


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/2684 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2684 [00:00<?, ? examples/s]

test set: 2684 items -> {'conjunction/affirmed': 607, 'disjunction/affirmed': 478, 'simple/affirmed': 1080, 'simple/negated': 519}


extracting: 100%|██████████| 2684/2684 [00:55<00:00, 48.54it/s]



=== Gemma-2-2B (L13) → Qwen2.5-1.5B (L17)  [stmt_origadapter] ===
  items differing at worst α: 0
  max |Δscore| 0.00e+00   max |ΔAUROC| 0.00e+00
  ceiling change (items): source +0, target +0
  strict overshoot: re-run +2.38 pp, saved +2.38 pp

✓ 1 re-swept run(s) match the saved sweeps (≤ 0 items at every α)


## Part 2 — statement-type Tables A–D (CPU)

Computed by `residuals.statement_type_tables` over the 12 full-test-set trained runs and compared with
`analysis_statement_types/*.csv`. Unlike the old Cell 10, the population is pinned rather than
globbed (the glob now also matches the 6 baselines), and the bootstrap is fixed at seed 0 with 1000
resamples. The tables are computed from the saved sweeps. If Part 1 re-swept all 12 full-test-set
runs, they are also recomputed from the re-run.

In [5]:
FULL_IDS = residuals.run_ids(residuals.TRAINED_VARIANTS)
sources = [("saved", SWEEPS)]
if all((OUT_DIR / p / "sweep_results.json").exists() for p in FULL_IDS):
    sources.append(("re-run", OUT_DIR))

stmt_tables = {}
for name, root in sources:
    tables = residuals.statement_type_tables(root)
    problems = residuals.compare_statement_tables(tables, SWEEPS / "analysis_statement_types")
    stmt_tables[name] = tables
    print(f"{name}:")
    for key, probs in problems.items():
        print(f"  {residuals.STATEMENT_TABLES[key]:<32} "
              + ("✓" if not probs else f"✗ {len(probs)} difference(s)"))
        for p in probs:
            print("     ", p)
    if any(problems.values()):
        raise SystemExit(f"REGRESSION — statement-type tables from the {name} sweeps changed")

out = OUT_DIR / "analysis_statement_types"
out.mkdir(parents=True, exist_ok=True)
for key, df in stmt_tables[sources[-1][0]].items():
    df.to_csv(out / f"{residuals.STATEMENT_TABLES[key]}.csv", index=False)
print("\n" + stmt_tables["saved"]["C"].to_string(index=False))

saved:
  tableA_per_run_detail            ✓
  tableB_trajectory_aggregate      ✓
  tableC_structural_consistency    ✓
  tableD_bootstrap_cis             ✓
  long_form_all_metrics            ✓

      group  runs plateau_holds collapses_at_alpha1
        all    12         11/12                0/12
conjunction    12         12/12                0/12
disjunction    12         11/12                6/12
    negated    12         12/12                5/12
     simple    12         12/12                1/12


## Part 3 — baselines and activation dumps (GPU, optional)

A port of the old Cell 11, run per trajectory: extract h_S/h_T once on the full test set, apply the
four trained adapters, build the two Mse8 baselines, α-sweep both, and dump everything for Table F.
Everything goes to `OUT_DIR`.

**Disclosure: the random-weight baseline is not reproducible.** The original Cell 11 built the
random MLP without seeding torch, so the published `stmt_randbase` runs and the `rand_out` arrays in
the dumps are one unseeded draw. Re-running produces a different random map. It is kept unseeded here
so the code matches what produced the published numbers. The claim it supports is about the kind of
result (≈0 strict overshoot, t_G gap erased), and Part 4 checks that claim rather than the digits.

The **shuffled** baseline is reproducible: its permutation comes from `default_rng(42)`, drawn once
per trajectory in the original order, so running a subset of trajectories still gets the same
permutations. The trained-adapter outputs are fixed weights.

**Adapter revisions.** Before any GPU work, the four adapters per trajectory are verified against the
published `stmt_*` payloads and pinned to those commits. Their commits are recorded in the baseline
payloads and in `<trajectory>.revisions.json` next to each dump.

In [6]:
baseline_checks = {}
if not RUN_BASELINES:
    print("RUN_BASELINES = False — skipping Part 3 (and Part 4)")
else:
    import gc, torch
    from datetime import datetime, timezone
    from representation_transfer.autoencoder import AutoEncoder
    from shared_truth import activations, probes, sweep

    activations.install_monkeypatch()
    records, items, _ = sweep.load_test_set(None)
    N_FULL = len(records)
    RNG = np.random.default_rng(42)                     # as Cell 11
    SEED_TAGS = [("_seed_42", "seed42"), ("_seed_43", "seed43"), ("_seed_44", "seed44")]
    DUMP_OUT = OUT_DIR / storage.DUMPS_DIR
    DUMP_OUT.mkdir(parents=True, exist_ok=True)
    chosen = residuals.TRAJECTORY_ADAPTERS if BASELINE_TRAJ is None else BASELINE_TRAJ
    ADAPTER_TAGS = (("orig", "stmt_origadapter"), ("seed42", "stmt_seed42"),
                    ("seed43", "stmt_seed43"), ("seed44", "stmt_seed44"))

    # the repos must still serve the adapters the published stmt_* runs used; pin to those commits
    ADAPTER_REVS = {}
    for b in chosen:
        s_, sl_, t_, tl_ = naming.parse_adapter_repo(b)
        for tag, variant in ADAPTER_TAGS:
            pl = json.load(open(SWEEPS / naming.pair_id(s_, sl_, t_, tl_, variant) / "sweep_results.json"))
            try:
                ADAPTER_REVS[(b, tag)] = storage.verify_adapter(pl)
            except storage.AdapterMismatchError as err:
                raise SystemExit(f"Part 3: {err}")
    print(f"adapters verified and pinned: {len(ADAPTER_REVS)} ({len(chosen)} trajectories x 4)")

    for base in residuals.TRAJECTORY_ADAPTERS:          # original order: the RNG stream depends on it
        perm = RNG.permutation(N_FULL)
        if base not in chosen:
            continue
        src, sl, tgt, tl = naming.parse_adapter_repo(base)
        label, slug = naming.display_pair_label(src, sl, tgt, tl), residuals.trajectory_slug(base)
        orig_pid = naming.pair_id(src, sl, tgt, tl, "stmt_origadapter")
        subfolder = json.load(open(SWEEPS / orig_pid / "sweep_results.json"))["adapter"]["subfolder"]
        print(f"\n=== {label} ===")

        src_tok, src_wrap = activations.load_model(src)
        tgt_tok, tgt_wrap = activations.load_model(tgt)
        tgt_probe, tgt_dirs = probes.load_probe(naming.probe_repo_for(tgt, tl))
        src_probe, src_dirs = probes.load_probe(naming.probe_repo_for(src, sl))

        # one extraction pass; the orig adapter is applied inline
        mapper0, cfg0 = activations.load_adapter(base, subfolder, revision=ADAPTER_REVS[(base, "orig")])
        c_src, c_tgt, c_map0, labels = activations.collect_activations(
            src_wrap, tgt_wrap, src_tok, tgt_tok,
            f"model.layers.{sl}", f"model.layers.{tl}", mapper0, records)
        labels = [int(l) for l in labels]
        H_S, H_T = np.concatenate(c_src, axis=0), np.concatenate(c_tgt, axis=0)
        outputs = {"orig": np.concatenate(c_map0, axis=0)}

        for suffix, tag in SEED_TAGS:                    # remaining adapters on cached h_S
            m, _ = activations.load_adapter(f"{base}{suffix}", subfolder, revision=ADAPTER_REVS[(base, tag)])
            with torch.no_grad():
                xs = torch.tensor(H_S, dtype=torch.float16, device="cuda")
                outputs[tag] = m(xs).float().cpu().numpy()
            del m

        # baseline 1: random-weight MLP, output norm matched to native h_T (unseeded, as published)
        rand_mapper = AutoEncoder(source_dim=cfg0["source_dim"], target_dim=cfg0["target_dim"],
                                  hidden_dim=cfg0["hidden_dim"]).to("cuda").half()
        rand_mapper.eval()
        with torch.no_grad():
            raw = rand_mapper(torch.tensor(H_S, dtype=torch.float16, device="cuda")).float().cpu().numpy()
        scale = float(np.linalg.norm(H_T, axis=1).mean() / max(np.linalg.norm(raw, axis=1).mean(), 1e-8))
        rand_out = raw * scale
        del rand_mapper

        # baseline 2: the orig adapter's real outputs, permuted across examples
        shuf_out = outputs["orig"][perm]

        ceilings, _ = sweep.native_ceilings(c_src, c_tgt, labels, src_probe, src_dirs, tgt_probe, tgt_dirs)
        for variant, out_arr, note in [
            ("stmt_randbase", rand_out, f"random-weight MLP, output-norm matched to native target (scale={scale:.3f})"),
            ("stmt_shufbase", shuf_out, "orig trained adapter outputs, example-permuted (seed 42)"),
        ]:
            c_base = [out_arr[i:i + 1].astype(np.float32) for i in range(len(labels))]
            swp = sweep.alpha_sweep(c_tgt, c_base, labels, tgt_probe, tgt_dirs)
            pid = naming.pair_id(src, sl, tgt, tl, variant)
            storage.save_payload(OUT_DIR, pid, {
                "pair_id": pid, "display_label": label,
                "source_model": src, "target_model": tgt, "source_layer": sl, "target_layer": tl,
                "variant": variant,
                "adapter": {"repo": base, "note": note, "revision": ADAPTER_REVS[(base, "orig")]},
                "probes": {"source": naming.probe_repo_for(src, sl), "target": naming.probe_repo_for(tgt, tl)},
                "n_test": len(labels), "ceiling": ceilings, "alpha_sweep": swp, "items": items,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
            print(f"  saved {pid}")

        np.savez_compressed(
            DUMP_OUT / f"{slug}.npz",
            h_S=H_S.astype(np.float16), h_T=H_T.astype(np.float16),
            **{f"map_{k}": v.astype(np.float16) for k, v in outputs.items()},
            rand_out=rand_out.astype(np.float16), shuf_out=shuf_out.astype(np.float16),
            t_G=tgt_dirs[0], t_P=tgt_dirs[1],
            labels=np.array(labels, np.int8),
            statement_type=np.array([it.get("statement_type") or "?" for it in items]),
            polarity=np.array([it.get("polarity") or "?" for it in items]),
        )
        (DUMP_OUT / f"{slug}.revisions.json").write_text(json.dumps(
            {tag: {"repo": base if tag == "orig" else f"{base}_seed_{tag[4:]}", "subfolder": subfolder,
                   "revision": ADAPTER_REVS[(base, tag)]} for tag, _ in ADAPTER_TAGS}, indent=2))

        # --- checks against the saved run ---
        shuf_pid = naming.pair_id(src, sl, tgt, tl, "stmt_shufbase")
        rand_pid = naming.pair_id(src, sl, tgt, tl, "stmt_randbase")
        shuf_new = json.load(open(OUT_DIR / shuf_pid / "sweep_results.json"))["alpha_sweep"]
        shuf_old = json.load(open(SWEEPS / shuf_pid / "sweep_results.json"))["alpha_sweep"]
        d = metrics.sweep_diff(shuf_new, shuf_old)
        saved_dump = np.load(SWEEPS / storage.DUMPS_DIR / f"{slug}.npz")
        new_dump = np.load(DUMP_OUT / f"{slug}.npz")
        arr_diff = {k: float(np.abs(new_dump[k].astype(np.float32) - saved_dump[k].astype(np.float32)).max())
                    for k in ("h_S", "h_T", "map_orig", "map_seed42", "map_seed43", "map_seed44", "shuf_out")}
        rand_so = metrics.strict_overshoot(json.load(open(OUT_DIR / rand_pid / "sweep_results.json"))["alpha_sweep"])[0]
        rand_so_saved = metrics.strict_overshoot(json.load(open(SWEEPS / rand_pid / "sweep_results.json"))["alpha_sweep"])[0]
        baseline_checks[slug] = {"shuf_max_items_differing": max(r["items_differing"] for r in d),
                                 "dump_max_abs_diff": arr_diff,
                                 "rand_strict_new": rand_so, "rand_strict_saved": rand_so_saved}
        print(f"  shuffled baseline vs saved: {baseline_checks[slug]['shuf_max_items_differing']} item(s) differ at worst α")
        print("  dump arrays vs saved, max |Δ|: " + ", ".join(f"{k} {v:.1e}" for k, v in arr_diff.items()))
        print(f"  random-weight strict overshoot: new draw {rand_so:+.2f} pp, published draw {rand_so_saved:+.2f} pp")

        del src_wrap, tgt_wrap, mapper0
        gc.collect(); torch.cuda.empty_cache()

    bad = {s: c["shuf_max_items_differing"] for s, c in baseline_checks.items()
           if c["shuf_max_items_differing"] > ITEM_TOL}
    if bad:
        raise SystemExit(f"REGRESSION — shuffled baseline differs by more than {ITEM_TOL} item(s): {bad}")
    print(f"\n✓ shuffled baselines reproduce on {len(baseline_checks)} trajectory(ies)")


=== Gemma-2-2B (L13) → Llama-3.2-3B (L12) ===


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

non_linear-gemma2b_to_qwen5b-13_to_17/mo(…): downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

non_linear-gemma2b_to_qwen5b-13_to_17/mo(…): reconstructing file:   0%|          |  0.00B / 28.9MB            

non_linear-gemma2b_to_qwen5b-13_to_17/mo(…): downloading bytes:           |  0.00B            

  saved gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_randbase
  saved gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_shufbase
  shuffled baseline vs saved: 0 item(s) differ at worst α
  dump arrays vs saved, max |Δ|: h_S 0.0e+00, h_T 0.0e+00, map_orig 0.0e+00, map_seed42 0.0e+00, map_seed43 0.0e+00, map_seed44 0.0e+00, shuf_out 0.0e+00
  random-weight strict overshoot: new draw +0.75 pp, published draw +0.00 pp

✓ shuffled baselines reproduce on 3 trajectory(ies)


## Part 4 — Tables E and F from the re-run (CPU, needs Part 3 on all 3 trajectories)

Tables E and F recomputed from the Part 3 outputs and compared with the published CSVs. Table F's
split draws from one RNG across all three dumps, so it needs all three. The random-weight rows are
reported separately: they come from a new unseeded draw and are expected to differ in value. Their
t_G gap ratio is checked against the claim (|ratio| < 0.15 against 1.0 native; the published draw
sits at −0.008 to +0.058). Their strict overshoot is not: a single draw ranges from 0.00 pp
(published) to +2.05 pp (a 2026-09-30 rerun), so it is judged against the seeded null in Part 5. Re-probe AUROCs use
`residuals.REPROBE_TOL`, as in the figures notebook.

In [8]:
RAND = {"random-weight", "rand_out"}
new_dumps = sorted((OUT_DIR / storage.DUMPS_DIR).glob("*.npz")) if RUN_BASELINES else []
if len(new_dumps) != 3:
    print("Part 4 needs Part 3 on all 3 trajectories — skipping")
else:
    def _load(pid):   # prefer the re-run, fall back to the saved sweep (e.g. an orig not re-swept)
        for root in (OUT_DIR, SWEEPS):
            f = root / pid / "sweep_results.json"
            if f.exists():
                return json.load(open(f))
    te_new = residuals.table_e(None, load=_load)
    tf_new = residuals.table_f(new_dumps)
    an = SWEEPS / "analysis_residuals"
    for t, rows, csv in (("E", te_new, "tableE_baseline_vs_trained.csv"),
                         ("F", tf_new, "tableF_residual_geometry.csv")):
        probs = residuals.compare_published(rows, an / csv, t)
        expected = [p for p in probs if any(f"'{r}')" in p for r in RAND)]
        real = [p for p in probs if p not in expected]
        print(f"Table {t}: {'✓' if not real else f'✗ {len(real)} difference(s)'}"
              f"   ({len(expected)} random-weight difference(s), expected)")
        for p in real:
            print("   ", p)
        if real:
            raise SystemExit(f"REGRESSION — Table {t} changed outside the random-weight rows")

    rand_e = [r for r in te_new if r["map"] == "random-weight"]
    rand_f = [r for r in tf_new if r["map"] == "rand_out"]
    print("\nrandom-weight (new draw), checked as claims:")
    for r in rand_e:
        print(f"  {r['trajectory']:<42} strict overshoot {r['strict_dAcc_taustar_pp']:+.2f} pp")
    for r in rand_f:
        print(f"  {r['trajectory']:<52} gap ratio t_G {r['gap_ratio_tG']:+.3f}")
    # published draw: -0.008 to +0.058, vs 1.0 native
    assert all(abs(r["gap_ratio_tG"]) < 0.15 for r in rand_f), "random-weight t_G gap not erased"

Table E: ✓   (7 random-weight difference(s), expected)
Table F: ✓   (18 random-weight difference(s), expected)

random-weight (new draw), checked as claims:
  Gemma-2-2B (L13) → Llama-3.2-3B (L12)      strict overshoot +2.05 pp
  Llama-3.2-3B (L12) → Gemma-2-2B (L13)      strict overshoot +0.34 pp
  Gemma-2-2B (L13) → Qwen2.5-1.5B (L17)      strict overshoot +0.75 pp
  gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12   gap ratio t_G -0.005
  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17  gap ratio t_G +0.026
  llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13   gap ratio t_G +0.028


## Part 5 — seeded random-weight null (GPU recommended; CPU works)

The published random-weight baseline is one unseeded draw (0.00 pp strict overshoot on all three
trajectories); a rerun drew +2.05, +0.34 and +0.75 pp. So this builds the distribution: `N_NULL` random
MLPs per trajectory, each constructed exactly as the old Cell 11 did (same `AutoEncoder` dims, fp16 on
GPU, output norm matched to native h_T over the full test set) with `torch.manual_seed(s)` before each.
Every map is scored on each eval in `EVALS`:

- `"full"`: the full test set (n=2684), calibrated against the saved `stmt_*` runs;
- `"simple"`: simple statements (n=1599), calibrated against the saved `truth_*` runs. This is the eval
  behind Table 7 and the multi-seed claims.

It reads h_S and h_T from the published dumps, so no model is loaded and Parts 1–4 need not run.
**Calibration** comes first, per eval: the batched sweeps from the dump are compared with the four
saved trained sweeps. Misaligned items (|Δscore| ≥ 0.05) or more than `CAL_ITEM_TOL` changed
predictions at any α stop the cell. A few threshold flips are tolerated and listed: the saved
simple-statement runs were extracted in a separate session, and fp16 kernels differ slightly across
GPUs. The first real run showed one flip, with |Δscore| 4.9e-3. A nonzero α=0 difference means the
native target activations themselves differ between sessions. Trained and null values are both
computed from the dump, so the p-values compare like with like either way; saved values are shown
alongside when they differ.

**Checkpoints.** The dump holds the adapter checkpoints of the `stmt_*` runs. If a saved run's recorded
adapter metrics differ, it used a different checkpoint of the same repo, so it is not used for
calibration. It is listed with ⚠, and its p-value tests the current checkpoint. This applies to the
floor pair's seeds 42–44 in the simple-statement (`truth_*`) runs: those adapters had fvu ≈ 0.038 and
cosine ≈ 0.69, against ≈ 0.005 and 0.72 for the checkpoints now in the repos.

**Reading it.** Each trained adapter gets a one-sided empirical p-value,
(1 + #{null ≥ trained}) / (1 + N). The smallest attainable value is 1/(N+1). Being above the null's
95th percentile is *not* the same as p < 0.05 (with 100 draws it corresponds to p ≤ 0.059), so the
summary counts p-values. It also Holm-corrects the original adapter's p across the three
trajectories. Saved to `OUT_DIR/random_weight_null_N{N_NULL}.json`. Seeds are 0..N−1, so a smaller run
is a prefix of a larger one. On CPU the MLP runs in fp32, so its draws differ from GPU fp16 ones.

In [13]:
# --- Part 5: seeded random-weight null (GPU recommended; reads the dumps, no model loading) ---
# N_NULL random-weight MLPs per trajectory, each built exactly like Cell 11's randbase (same
# AutoEncoder dims, fp16 on GPU, output norm matched to native h_T over the full test set) but
# with torch seeded per draw. Each map is scored on every eval in EVALS, so the nulls for the
# two evals come from the same draws.
import torch
from huggingface_hub import hf_hub_download
from representation_transfer.autoencoder import AutoEncoder
from shared_truth import probes, sweep

N_NULL = 1000                  # seeded draws per trajectory: seeds 0..N_NULL-1
EVALS  = ("full", "simple")    # "full": full test set, vs the stmt_* runs (n=2684)
                               # "simple": simple statements, vs the truth_* runs (n=1599) --
                               #           the eval behind Table 7 and the multi-seed claims
ALPHA  = 0.05
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32   # Cell 11 ran the MLP in fp16 on GPU
SAVED_PREFIX = {"full": "stmt_", "simple": "truth_"}
TRAINED = [("map_orig", "origadapter"), ("map_seed42", "seed42"),
           ("map_seed43", "seed43"), ("map_seed44", "seed44")]
MAX_DSCORE = 0.05   # catches item misalignment (~0.5+); cross-session fp16 drift was ~5e-3
CAL_ITEM_TOL = 3    # predictions per α allowed to differ from a saved run (see calibration note below)

def _sweep_mapped(mapped, H_T, labels, probe, dirs):
    return sweep.alpha_sweep_with_adapter(lambda _: mapped, None, H_T, probe, dirs, labels)

def _strict(mapped, H_T, labels, probe, dirs):
    return metrics.strict_overshoot(_sweep_mapped(mapped, H_T, labels, probe, dirs))[0]

def _holm(pvals):   # Holm-adjusted p-values for a dict of p-values
    order = sorted(pvals, key=pvals.get)
    out, running = {}, 0.0
    for i, k in enumerate(order):
        running = max(running, min(1.0, (len(order) - i) * pvals[k]))
        out[k] = running
    return out

null_results = {e: {} for e in EVALS}
for base in residuals.TRAJECTORY_ADAPTERS:
    src, sl, tgt, tl = naming.parse_adapter_repo(base)
    label, slug = naming.display_pair_label(src, sl, tgt, tl), residuals.trajectory_slug(base)
    d = np.load(storage.ensure_local(storage.DUMPS_DIR, f"{slug}.npz"))
    H_S, H_T = d["h_S"].astype(np.float32), d["h_T"].astype(np.float32)
    labels = d["labels"].astype(int)
    masks = {"full": np.ones(len(labels), bool), "simple": d["statement_type"] == "simple"}
    probe, dirs = probes.load_probe(naming.probe_repo_for(tgt, tl))
    pid = lambda v: naming.pair_id(src, sl, tgt, tl, v)
    _pl = {}
    def saved_payload(v):
        if v not in _pl:
            _pl[v] = json.load(open(SWEEPS / pid(v) / "sweep_results.json"))
        return _pl[v]
    ckpt = lambda v: (saved_payload(v)["adapter"].get("config") or {}).get("metrics")
    ad = saved_payload("stmt_origadapter")["adapter"]
    cfg = ad.get("config") or json.load(open(hf_hub_download(base, "config.json", subfolder=ad["subfolder"])))
    print(f"\n=== {label} ===")

    # calibration: batched sweeps from the dump must reproduce each eval's saved sweeps. Misalignment
    # or a real change stops the cell. A few threshold flips are tolerated and reported: saved runs
    # extracted in another session (e.g. another GPU) can differ at the fp16 level. Trained and null
    # values below are both computed from the dump, so the p-values stay like-for-like either way.
    # The dump holds the checkpoints of the stmt_* runs (same session). A saved run whose recorded
    # adapter metrics differ used a different checkpoint of the same repo, so it cannot calibrate the
    # dump; it is reported, and the dump-based value (the current checkpoint) is what gets tested.
    trained, published, saved_strict, cal = {}, {}, {}, {}
    for e in EVALS:
        mk, trained[e], saved_strict[e], max_dscore, a0_dscore, exact = masks[e], {}, {}, 0.0, 0.0, True
        n_items, notes, mismatched = int(mk.sum()), [], []
        for key, tag in TRAINED:
            saved = saved_payload(SAVED_PREFIX[e] + tag)["alpha_sweep"]
            n_saved = len(saved[0]["true_scores"]) + len(saved[0]["false_scores"])
            if n_saved != mk.sum():
                raise SystemExit(f"[{e}] {pid(SAVED_PREFIX[e] + tag)} has {n_saved} items, "
                                 f"the dump's {e} subset has {int(mk.sum())}")
            rows = _sweep_mapped(d[key].astype(np.float32)[mk], H_T[mk], labels[mk], probe, dirs)
            diff = metrics.sweep_diff(rows, saved)
            so, so_saved = metrics.strict_overshoot(rows)[0], metrics.strict_overshoot(saved)[0]
            this_ckpt, dump_ckpt = ckpt(SAVED_PREFIX[e] + tag), ckpt("stmt_" + tag)
            if this_ckpt != dump_ckpt:
                fvu = lambda m: f"{m['fvu']:.4f}" if m and "fvu" in m else "?"
                mismatched.append(tag)
                notes.append(f"{tag}: ⚠ saved run used a different checkpoint (fvu {fvu(this_ckpt)} vs "
                             f"{fvu(dump_ckpt)} in the dump) — not calibrated; strict {so:+.2f} (dump, "
                             f"current checkpoint) vs {so_saved:+.2f} (saved, other checkpoint)")
                trained[e][tag], saved_strict[e][tag] = so, so_saved
                continue
            items = max(r["items_differing"] for r in diff)
            dscore = max(r["max_abs_dscore"] for r in diff)
            max_dscore, exact = max(max_dscore, dscore), exact and all(r["items_exact"] for r in diff)
            a0_dscore = max(a0_dscore, diff[0]["max_abs_dscore"])
            flips = [(r["alpha"], r["items_differing"]) for r in diff if r["items_differing"]]
            if (dscore > MAX_DSCORE or items > CAL_ITEM_TOL
                    or abs(so - so_saved) > CAL_ITEM_TOL * 100 / n_items + 1e-9):
                print(f"  [{e}] {tag}: per-α (items differing, max |Δscore|): "
                      + ", ".join(f"α={r['alpha']}: {r['items_differing']}, {r['max_abs_dscore']:.1e}" for r in diff))
                raise SystemExit(f"[{e}] calibration failed for {pid(SAVED_PREFIX[e] + tag)}: "
                                 f"{items} item(s) differ, max |Δscore| {dscore:.1e}, strict "
                                 f"{so:+.4f} vs saved {so_saved:+.4f} — the null would not be comparable")
            if flips or abs(so - so_saved) > 1e-9:
                notes.append(f"{tag}: {', '.join(f'{k} item(s) at α={a}' for a, k in flips) or 'no flips'}; "
                             f"strict {so:+.2f} (dump) vs {so_saved:+.2f} (saved)")
            trained[e][tag], saved_strict[e][tag] = so, so_saved
        # the published draw, from its fp16 copy in the dump (saved sweep used fp32, so approximate)
        published[e] = _strict(d["rand_out"].astype(np.float32)[mk], H_T[mk], labels[mk], probe, dirs)
        cal[e] = {"exact": not notes, "notes": notes, "checkpoint_mismatch": mismatched,
                  "max_abs_dscore": max_dscore, "alpha0_max_abs_dscore": a0_dscore}
        status = ("✓ exact" if not notes else "≈ within tolerance" if not mismatched
                  else f"⚠ {len(mismatched)} run(s) on a different checkpoint; the rest "
                       + ("exact" if len(notes) == len(mismatched) else "within tolerance"))
        print(f"  [{e}] n={n_items}  calibration {status}: {4 - len(mismatched)} trained runs vs saved "
              f"({'per-item preds' if exact else 'per-α accuracy + scores'}), "
              f"max |Δscore| {max_dscore:.1e}, at α=0 (native h_T only) {a0_dscore:.1e}")
        for note in notes:
            print(f"      {note}")

    # the null: one map per seed, scored on every eval
    xs = torch.tensor(H_S, dtype=DTYPE, device=DEVICE)
    null = {e: [] for e in EVALS}
    for s in range(N_NULL):
        torch.manual_seed(s)
        m = AutoEncoder(source_dim=cfg["source_dim"], target_dim=cfg["target_dim"],
                        hidden_dim=cfg["hidden_dim"]).to(DEVICE).to(DTYPE).eval()
        with torch.no_grad():
            raw = m(xs).float().cpu().numpy()
        scale = float(np.linalg.norm(H_T, axis=1).mean() / max(np.linalg.norm(raw, axis=1).mean(), 1e-8))
        mapped = raw * scale
        for e in EVALS:
            mk = masks[e]
            null[e].append(_strict(mapped[mk], H_T[mk], labels[mk], probe, dirs))
        del m
        if (s + 1) % 250 == 0:
            print(f"  {s + 1}/{N_NULL} draws")
    del xs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    for e in EVALS:
        arr = np.array(null[e])
        p = {t: (1 + int((arr >= v).sum())) / (1 + len(arr)) for t, v in trained[e].items()}
        null_results[e][label] = {
            "n_items": int(masks[e].sum()), "n_null": len(arr), "null": arr.tolist(),
            "median": float(np.median(arr)), "p95": float(np.percentile(arr, 95)), "max": float(arr.max()),
            "frac_positive": float((arr > 0).mean()), "published_draw_from_dump": published[e],
            "trained": trained[e], "saved_strict": saved_strict[e], "calibration": cal[e], "p_value": p,
        }
        r = null_results[e][label]
        print(f"  [{e}] null: median {r['median']:+.2f}  p95 {r['p95']:+.2f}  max {r['max']:+.2f} pp, "
              f"> 0 in {r['frac_positive']:.0%} of draws;  published draw {published[e]:+.2f} pp")
        for t, v in trained[e].items():
            sv = saved_strict[e][t]
            tagnote = ("" if abs(v - sv) < 1e-9 and t not in cal[e]["checkpoint_mismatch"]
                       else f" (saved {sv:+.2f}{', other checkpoint' if t in cal[e]['checkpoint_mismatch'] else ''})")
            print(f"      {t:<12} {v:+.2f} pp{tagnote}   p = {p[t]:.4f}   {'*' if p[t] < ALPHA else ''}")

# --- summary: p-values, not the 95th percentile (the +1 correction makes those differ) ---
holm_orig = {}
ck = [(e, lab, r["calibration"]["checkpoint_mismatch"]) for e in EVALS for lab, r in null_results[e].items()
      if r["calibration"]["checkpoint_mismatch"]]
for e, lab, tags in ck:
    print(f"\n⚠ [{e}] {lab}: saved runs for {', '.join(tags)} used a different adapter checkpoint than the "
          f"dump; their p-values test the current checkpoint, not the published {e}-eval runs")
print(f"\nsmallest attainable p with N_NULL={N_NULL}: {1 / (N_NULL + 1):.4f};  * = p < {ALPHA}, uncorrected")
for e in EVALS:
    res = null_results[e]
    n_runs = sum(len(r["trained"]) for r in res.values())
    n_sig = sum(pv < ALPHA for r in res.values() for pv in r["p_value"].values())
    holm_orig[e] = _holm({lab: r["p_value"]["origadapter"] for lab, r in res.items()})
    orig_sig = sum(pv < ALPHA for pv in holm_orig[e].values())
    print(f"[{e}] trained runs beating the random-weight null at p < {ALPHA}: {n_sig}/{n_runs} (uncorrected);"
          f"  orig, Holm-corrected across the {len(res)} trajectories: {orig_sig}/{len(res)}")
    for lab, pv in holm_orig[e].items():
        print(f"      {lab:<42} orig p = {res[lab]['p_value']['origadapter']:.4f}  Holm {pv:.4f}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
null_path = OUT_DIR / f"random_weight_null_N{N_NULL}.json"   # seeds 0..N-1: a smaller run is a prefix of a larger one
json.dump({"n_null": N_NULL, "seeds": [0, N_NULL - 1], "evals": list(EVALS), "alpha": ALPHA,
           "device": DEVICE, "dtype": str(DTYPE),
           "p_value": "one-sided, (1 + #{null >= trained}) / (1 + N_NULL)",
           "results": null_results, "holm_orig": holm_orig, "provenance": storage.provenance()},
          open(null_path, "w"), indent=2)
print(f"saved → {null_path}")


=== Gemma-2-2B (L13) → Llama-3.2-3B (L12) ===
  [full] n=2684  calibration ✓ exact: 4 trained runs vs saved (per-item preds), max |Δscore| 1.3e-04, at α=0 (native h_T only) 3.6e-07
  [simple] n=1599  calibration ≈ within tolerance: 4 trained runs vs saved (per-α accuracy + scores), max |Δscore| 4.9e-03, at α=0 (native h_T only) 4.9e-03
      origadapter: 1 item(s) at α=0.3, 1 item(s) at α=0.9, 1 item(s) at α=1.0; strict +9.01 (dump) vs +9.01 (saved)
      seed42: 1 item(s) at α=0.4, 2 item(s) at α=1.0; strict +7.44 (dump) vs +7.44 (saved)
      seed43: 1 item(s) at α=1.0; strict +5.63 (dump) vs +5.57 (saved)
      seed44: 1 item(s) at α=0.3, 2 item(s) at α=0.8, 1 item(s) at α=1.0; strict +5.38 (dump) vs +5.32 (saved)
  250/1000 draws
  500/1000 draws
  750/1000 draws
  1000/1000 draws
  [full] null: median +0.04  p95 +3.58  max +5.96 pp, > 0 in 51% of draws;  published draw +0.00 pp
      origadapter  +5.59 pp   p = 0.0090   *
      seed42       +4.43 pp   p = 0.0340   *
      seed43 

In [14]:
# --- manifest ---------------------------------------------------------------
manifest = {
    "run_set": RUN_SET, "rerun_ids": RERUN, "item_tol": ITEM_TOL,
    "part1": checks, "part3": baseline_checks,
    "tables_ad_sources": [n for n, _ in sources],
    "provenance": storage.provenance(),
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
json.dump(manifest, open(OUT_DIR / "rerun_manifest.json", "w"), indent=2, default=float)
print(f"manifest → {OUT_DIR / 'rerun_manifest.json'}")

manifest → /content/adapter_rerun/rerun_manifest.json
